# Stage 3: Model Development and Evaluation
This notebook focuses on training the Basel component models: PD, EAD, and LGD.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import classification_report, roc_auc_score, r2_score, mean_absolute_error

### 1. Data Preparation

In [ ]:
# Note: Using the final preprocessed dataframe from Stage 2
# For this notebook, we simulate the loading process
df_final = pd.read_csv('/content/final_preprocessed_data.csv') # Assuming previous export

X = df_final.drop(columns=['Loan_Status'])
y = df_final['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print('Feature Preview (First 5 rows):')
display(X_train.head())

### 2. Probability of Default (PD) Model

In [ ]:
pd_model = LogisticRegression(max_iter=1000, random_state=42)
pd_model.fit(X_train, y_train)

y_prob_pd = pd_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_prob_pd)
print(f'PD Model ROC-AUC: {auc:.4f}')

# Feature Importance
importance = pd.DataFrame({'Feature': X_train.columns, 'Weight': pd_model.coef_[0]}).sort_values(by='Weight')
plt.figure(figsize=(10, 6))
sns.barplot(data=importance, x='Weight', y='Feature')
plt.title('PD Model: Feature Importance')
plt.show()

### 3. Exposure at Default (EAD) Model

In [ ]:
# Target for EAD is the LoanAmount
y_train_ead = X_train['LoanAmount']
y_test_ead = X_test['LoanAmount']
X_train_ead = X_train.drop(columns=['LoanAmount'])
X_test_ead = X_test.drop(columns=['LoanAmount'])

ead_model = LinearRegression()
ead_model.fit(X_train_ead, y_train_ead)
y_pred_ead = ead_model.predict(X_test_ead)

print(f'EAD Model R2 Score: {r2_score(y_test_ead, y_pred_ead):.4f}')

# Actual vs Predicted Scatter Plot
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test_ead, y=y_pred_ead)
plt.plot([y_test_ead.min(), y_test_ead.max()], [y_test_ead.min(), y_test_ead.max()], 'r--')
plt.title('EAD Model: Actual vs Predicted Exposure')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.show()

### 4. Loss Given Default (LGD) Proxy Model

In [ ]:
np.random.seed(42)
y_train_lgd = np.random.uniform(0.2, 0.8, size=len(X_train))
y_test_lgd = np.random.uniform(0.2, 0.8, size=len(X_test))

lgd_model = LinearRegression()
lgd_model.fit(X_train, y_train_lgd)
y_pred_lgd = lgd_model.predict(X_test)

print(f'LGD Proxy R2 Score: {r2_score(y_test_lgd, y_pred_lgd):.4f}')